# Tesseract OCR on PDFs

Rasterizes each page with **pdf2image** (Poppler), then runs **Tesseract**.

## What runs inside the virtual environment

Python packages (`pytesseract`, `pdf2image`, Jupyter, etc.) are installed into **this project’s `.venv`** via the next cells (`pip install -r requirements.txt` uses whatever kernel you selected—pick `.venv` so packages land there).

## What still needs Homebrew (not inside the venv)

Tesseract and Poppler are **system programs**; they are not Python wheels. Install once on your Mac:

```bash
brew install tesseract poppler
# optional: extra languages (e.g. Swahili)
brew install tesseract-lang
```

## One-time: create the `.venv` (Python 3.11+)

Apple’s default `python3` is often 3.9. Prefer Homebrew 3.11+:

```bash
cd /Users/ronaldwahome/Documents/test_ocr_inputs
/opt/homebrew/bin/python3.11 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install -r requirements.txt
python -m ipykernel install --user --name=test_ocr_inputs --display-name="Python (test_ocr_inputs)"
```

In Jupyter or Cursor, choose the kernel **Python (test_ocr_inputs)** or the interpreter `.venv/bin/python`, then run all cells below.

Place `.pdf` files anywhere under this project folder; they appear in the dropdown after you run the UI cell.

In [4]:
# Install Python deps into the *current* kernel environment (your .venv if that kernel is selected).
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [1]:
import shutil
import subprocess
import sys
from pathlib import Path

NOTEBOOK_ROOT = Path.cwd().resolve()
VENV_PY = NOTEBOOK_ROOT / ".venv" / "bin" / "python"
in_venv = sys.prefix.endswith(".venv") or (sys.base_prefix != sys.prefix)

print(f"Python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
print(f"Executable: {sys.executable}")
if VENV_PY.is_file() and Path(sys.executable).resolve() != VENV_PY.resolve():
    print(
        f"Tip: select the project venv kernel so packages and OCR use: {VENV_PY}"
    )
elif in_venv:
    print("Virtual environment:", sys.prefix)

ok = True
if not shutil.which("tesseract"):
    print("Missing tesseract → run: brew install tesseract")
    ok = False
else:
    p = subprocess.run(
        ["tesseract", "--version"], capture_output=True, text=True, timeout=10
    )
    line = (p.stdout or p.stderr or "").strip().splitlines()
    print("Tesseract:", line[0] if line else "(unknown)")

if not shutil.which("pdfinfo"):
    print("Missing poppler (pdfinfo) → run: brew install poppler")
    ok = False
else:
    p = subprocess.run(
        ["pdfinfo", "-v"], capture_output=True, text=True, timeout=10
    )
    line = (p.stdout or p.stderr or "").strip().splitlines()
    print("Poppler:", line[0] if line else "pdfinfo ok")

if not ok:
    print("\nInstall the items above, then restart the kernel and re-run from the top.")

Python 3.11.15
Executable: /Users/ronaldwahome/Documents/test_ocr_inputs/.venv/bin/python
Virtual environment: /Users/ronaldwahome/Documents/test_ocr_inputs/.venv
Missing tesseract → run: brew install tesseract
Missing poppler (pdfinfo) → run: brew install poppler

Install the items above, then restart the kernel and re-run from the top.


In [ ]:
from pathlib import Path

import ipywidgets as widgets
import pytesseract
from IPython.display import display, clear_output

from ocr_tesseract_pdf import (
    find_pdfs,
    ocr_pdf,
    page_count,
    project_root,
    tesseract_version,
)

ROOT = project_root()
print(f"Project root: {ROOT}")
try:
    print(f"Tesseract (pytesseract): {tesseract_version()}")
except pytesseract.TesseractNotFoundError:
    print(
        "Tesseract: not on PATH — install with `brew install tesseract`, then restart the kernel."
    )

In [ ]:
pdfs = find_pdfs(ROOT)

pdf_dropdown = widgets.Dropdown(
    options=[(p.relative_to(ROOT).as_posix(), str(p)) for p in pdfs] if pdfs else [("(no PDFs found)", "")],
    description="PDF:",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "initial"},
)

dpi_slider = widgets.IntSlider(value=200, min=72, max=400, step=1, description="DPI:")
lang_text = widgets.Text(value="eng", description="Tesseract lang:", layout=widgets.Layout(width="50%"))

page_from = widgets.BoundedIntText(value=1, min=1, max=9999, description="From page:")
page_to = widgets.BoundedIntText(value=1, min=1, max=9999, description="To page:")

run_btn = widgets.Button(description="Run OCR", button_style="primary")
out = widgets.Output()


def _sync_page_bounds(*_):
    path = pdf_dropdown.value
    if not path or not Path(path).exists():
        return
    try:
        n = page_count(path)
    except Exception:
        return
    page_from.max = n
    page_to.max = n
    page_from.value = min(page_from.value, n)
    page_to.value = n


def _on_pdf_change(change):
    _sync_page_bounds()


pdf_dropdown.observe(_on_pdf_change, names="value")
_sync_page_bounds()


def _run_clicked(_):
    path = pdf_dropdown.value
    out.clear_output()
    with out:
        if not path or not Path(path).is_file():
            print("Add PDFs under the project folder and re-run the cell that builds the dropdown.")
            return
        fp, lp = int(page_from.value), int(page_to.value)
        if fp > lp:
            print("From page must be <= To page.")
            return
        dpi = int(dpi_slider.value)
        lang = lang_text.value.strip() or "eng"
        print(f"OCR: {Path(path).name}  pages {fp}-{lp}  DPI={dpi}  lang={lang}\n")
        try:
            results = ocr_pdf(path, dpi=dpi, lang=lang, first_page=fp, last_page=lp)
        except Exception as e:
            print(f"Error: {e}")
            return
        for page_num, text, img in results:
            print("=" * 60)
            print(f"PAGE {page_num}")
            print("=" * 60)
            display(img)
            print(text if text.strip() else "(no text detected)")
            print()


run_btn.on_click(_run_clicked)

ui = widgets.VBox(
    [
        pdf_dropdown,
        widgets.HBox([dpi_slider, lang_text]),
        widgets.HBox([page_from, page_to]),
        run_btn,
        out,
    ]
)
display(ui)

if not pdfs:
    print(f"No PDFs under {ROOT}. Copy some in and re-run this cell.")